# 📊 Exploratory Data Analysis (EDA)
## Real Estate Valuation Dataset

Comprehensive analysis of 12,814 real estate properties from Supabase

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

# Add parent directory to path
sys.path.insert(0, str(Path('.').resolve().parent))
from pipeline.supabase_handler import fetch_csv_from_supabase

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ Imports successful!')

## 1. Load Data from Supabase

In [ ]:
print('Loading data from Supabase...')
df = fetch_csv_from_supabase()
print(f'✅ Loaded {len(df)} records')
print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
df.head()

## 2. Data Overview

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\n' + '='*60)
print('Missing Values:')
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': missing[missing > 0].index,
    'Count': missing[missing > 0].values,
    'Percentage': missing_pct[missing_pct > 0].values
})
print(missing_df.to_string(index=False))

## 3. Price Analysis

In [ ]:
print('\n📈 PRICE STATISTICS:')
print(df['price_vnd'].describe())
print(f'\nPrice range: {df["price_vnd"].min():,.0f} - {df["price_vnd"].max():,.0f} VND')
print(f'Median price: {df["price_vnd"].median():,.0f} VND')

In [ ]:
# Price distribution visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Histogram
axes[0, 0].hist(df['price_vnd'].dropna(), bins=60, edgecolor='black', color='skyblue', alpha=0.7)
axes[0, 0].set_title('Price Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Price (VND)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(alpha=0.3)

# Log scale
axes[0, 1].hist(np.log10(df['price_vnd'].dropna()), bins=60, edgecolor='black', color='orange', alpha=0.7)
axes[0, 1].set_title('Price Distribution (Log10 Scale)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Log10(Price)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(alpha=0.3)

# Boxplot
axes[1, 0].boxplot(df['price_vnd'].dropna(), vert=True)
axes[1, 0].set_title('Price Box Plot', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Price (VND)')
axes[1, 0].grid(alpha=0.3)

# CDF
sorted_price = np.sort(df['price_vnd'].dropna())
cdf = np.arange(1, len(sorted_price) + 1) / len(sorted_price)
axes[1, 1].plot(sorted_price, cdf, linewidth=2.5, color='green')
axes[1, 1].set_title('Cumulative Distribution Function', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Price (VND)')
axes[1, 1].set_ylabel('Cumulative Probability')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('output/01_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 01_price_distribution.png')

## 4. Feature Correlation Analysis

In [ ]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_df = df[numeric_cols].copy()
numeric_df = numeric_df.replace({np.inf: np.nan, -np.inf: np.nan})

# Correlation with price
correlations = numeric_df.corr()['price_vnd'].sort_values(ascending=False)
print('\n📊 TOP 20 CORRELATED FEATURES WITH PRICE:')
print(correlations.head(20))

In [ ]:
# Correlation heatmap
top_features = correlations.abs().nlargest(15).index.tolist()

fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = numeric_df[top_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, cbar_kws={'label': 'Correlation'},
            square=True, linewidths=0.5)
ax.set_title('Correlation Matrix (Top 15 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/02_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 02_correlation_heatmap.png')

## 5. Geographic Analysis - Price Heatmap

In [ ]:
# Filter valid geographic data
valid_geo = df.dropna(subset=['lat', 'lon', 'price_vnd'])
print(f'\n📍 Geographic Analysis:')
print(f'Valid records with coordinates: {len(valid_geo)}')
print(f'Latitude range: {valid_geo["lat"].min():.4f} - {valid_geo["lat"].max():.4f}')
print(f'Longitude range: {valid_geo["lon"].min():.4f} - {valid_geo["lon"].max():.4f}')

In [ ]:
# Price heatmap
fig, ax = plt.subplots(figsize=(14, 11))
scatter = ax.scatter(valid_geo['lon'], valid_geo['lat'],
                    c=valid_geo['price_vnd'], cmap='viridis',
                    s=50, alpha=0.6, edgecolors='black', linewidth=0.3)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Price Heatmap - Geographic Distribution (Ho Chi Minh City)', 
            fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Price (VND)', fontsize=11)
plt.tight_layout()
plt.savefig('output/03_price_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 03_price_heatmap.png')

## 6. Feature Distribution Analysis

In [ ]:
# Key features distributions
key_features = ['area_m2', 'num_floors', 'num_bedrooms', 'road_width_m', 'distance_to_center_km']
existing_features = [f for f in key_features if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feature in enumerate(existing_features):
    axes[idx].hist(df[feature].dropna(), bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].set_title(f'{feature} Distribution', fontweight='bold')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(alpha=0.3)

# Hide unused subplots
for idx in range(len(existing_features), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('output/04_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 04_feature_distributions.png')

## 7. POI Features Analysis

In [ ]:
# POI features
poi_features = [f for f in df.columns if 'count' in f.lower()]
print(f'\n📍 POI FEATURES STATISTICS:')
for feature in poi_features[:8]:  # Show first 8
    print(f'{feature}: mean={df[feature].mean():.2f}, max={df[feature].max():.0f}')

In [ ]:
# POI correlation with price
poi_count_features = [f for f in df.columns if 'count' in f.lower()]
poi_corr = df[poi_count_features + ['price_vnd']].corr()['price_vnd'].sort_values(ascending=False)[1:]

fig, ax = plt.subplots(figsize=(10, 6))
poi_corr.plot(kind='barh', ax=ax, color='teal')
ax.set_title('POI Features Correlation with Price', fontsize=13, fontweight='bold')
ax.set_xlabel('Correlation Coefficient')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('output/05_poi_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 05_poi_correlation.png')

## 8. Summary Statistics

In [ ]:
# Create feature summary
feature_summary = pd.DataFrame({
    'Column': numeric_cols,
    'Missing': [df[col].isnull().sum() for col in numeric_cols],
    'Type': [df[col].dtype for col in numeric_cols],
    'Mean': [df[col].mean() for col in numeric_cols],
    'Std': [df[col].std() for col in numeric_cols],
    'Min': [df[col].min() for col in numeric_cols],
    'Max': [df[col].max() for col in numeric_cols]
})

feature_summary.to_csv('output/feature_summary.csv', index=False)
print('\n📊 Feature Summary:')
print(feature_summary.to_string(index=False))
print('\n✅ Saved: feature_summary.csv')

## 9. Key Insights & Conclusions

In [ ]:
print("\n" + "="*70)
print("✅ EDA COMPLETE - KEY INSIGHTS")
print("="*70)
print(f"""
📊 DATASET OVERVIEW:
  • Total records: {len(df):,}
  • Features: {len(df.columns)}
  • Missing values: {df.isnull().sum().sum():,}

💰 PRICE ANALYSIS:
  • Mean price: {df['price_vnd'].mean():,.0f} VND
  • Median price: {df['price_vnd'].median():,.0f} VND
  • Price std dev: {df['price_vnd'].std():,.0f} VND
  • Range: {df['price_vnd'].min():,.0f} - {df['price_vnd'].max():,.0f} VND

🎯 TOP PREDICTIVE FEATURES:
  1. Area (area_m2): {correlations['area_m2']:.3f}
  2. Width (width_m): {correlations['width_m']:.3f}
  3. Road width (road_width_m): {correlations['road_width_m']:.3f}
  4. Bedrooms (num_bedrooms): {correlations['num_bedrooms']:.3f}
  5. Supermarket count (supermarket_count_3km): {correlations['supermarket_count_3km']:.3f}

📍 GEOGRAPHIC DISTRIBUTION:
  • Valid coordinates: {len(valid_geo):,} properties
  • Average distance to center: {df['distance_to_center_km'].mean():.2f} km

⚠️  DATA QUALITY NOTES:
  • High missing values: width_m (1282), length_m (1686), nearest_metro_km (5605)
  • Consider handling outliers in price and area fields
  • Geographic clustering in Ho Chi Minh City

➡️  NEXT STEPS:
  1. Feature engineering & preprocessing
  2. Train ML model (target: MAPE < 10%)
  3. Model evaluation on test set
  4. Build BI dashboard
  5. Deploy web application
""")
print("="*70)